# அத்தியாயம் 7: உரையாடல் செயலிகளைக் கட்டமைத்தல்
## Microsoft Foundry மாதிரிகள் API விரைவான துவக்கம்

இந்த நோட்புக் [Azure OpenAI எடுத்துக்காட்டு சேகரிப்பில்](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) இருந்து பெற்று எடுக்கப்பட்டுள்ளது, இதில் [Azure OpenAI](notebook-azure-openai.ipynb) சேவைகளை அணுகும் நோட்புக்குகள் உள்ளன.

> **குறிப்பு:** GitHub மாதிரிகள் 2026 ஜூலை மாத இறுதியில் முடிவடைகிறது. இந்த நோட்புக் இப்போது அதே இலவச முயற்சி மாதிரி பட்டியலை மற்றும் Azure AI Inference SDK அனுபவத்தை வழங்கும் [Microsoft Foundry மாதிரிகளை](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) குறிக்கிறது.


# மேலோட்டம்  
"பெரிய மொழி மாதிரிகள் என்பது கட்டத்தை கட்டமாக மேப்பிங் செய்யும் செயல்பாடுகளாகும். ஒரு உள்ளீட்டு கட்டத்தை வழங்கும் போது, ஒரு பெரிய மொழி மாதிரி அடுத்ததாக வரும் கட்டையை கணிக்க முயற்சிக்கிறது"(1). இந்த "விரைவு ஆரம்ப" நோட்புக் பயனாளிகளுக்கு உயர்தர LLM கருத்துக்களுக்கு, AML உடன் துவங்க தேவையான முக்கிய பெட்டகத் தேவைகளுக்கு, ஒரு மென்மையான தூண்டுதல் வடிவமைப்பிற்கு அறிமுகத்தை மற்றும் பல்வேறு பயன்பாட்டு எடுத்துக்காட்டுகளை அறிமுகப்படுத்திவிடும். 


## உள்ளடக்க பட்டியல்  

[கண்ணோட்டம்](#overview)  
[OpenAI சேவையை எப்படி பயன்படுத்துவது](#how-to-use-openai-service)  
[1. உங்கள் OpenAI சேவையை உருவாக்குதல்](#1.-creating-your-openai-service)  
[2. நிறுவல்](#2.-installation)    
[3. அங்கீகாரங்கள்](#3.-credentials)  

[பயன்பாட்டு வழக்குகள்](#use-cases)    
[1. உரை சுருக்கம் செய்தல்](#1.-summarize-text)  
[2. உரையை வகைப்படுத்துதல்](#2.-classify-text)  
[3. புதிய தயாரிப்பு பெயர்களை உருவாக்குதல்](#3.-generate-new-product-names)  
[4. வகைப்படுத்தியலை நுண்ணறிவு செய்வது](#4.fine-tune-a-classifier)  

[குறிப்புகள்](#references)


### உங்கள் முதல் உறுதியை உருவாக்கவும்  
இந்த சிறிய பயிற்சி, Microsoft Foundry Models இல் "சுருக்கக்கூறும்" என்கிற எளிய பணிக்கான ஒரு மாதிரிக்கு உறுதிகளை சமர்ப்பிப்பதற்கான அடிப்படைக் குறிப்பு வழங்கும்.  


**படிகள்**:  
1. உங்கள் பைதான் சூழலில் `azure-ai-inference` நூலகத்தை நிறுவிக்கொள்ளுங்கள், நீங்கள் இதனை ஏற்கனவே செய்யாமல் இருக்கின்.  
2. நிலையான உதவி நூலகங்களை ஏற்று, Microsoft Foundry Models அங்கீகாரங்களை அமைக்கவும்.  
3. உங்கள் பணிக்கு ஏற்ப ஒரு மாதிரியை தேர்வு செய்யவும்  
4. மாதிரிக்கான ஒரு எளிய உறுதியை உருவாக்கவும்  
5. உங்கள் கோரிக்கையை மாதிரி APIக்கு சமர்ப்பிக்கவும்!  


### 1. `azure-ai-inference` ஐ நிறுவுங்கள்


In [ ]:
%pip install azure-ai-inference

### 2. உதவி நூலகங்களை இறக்குமதி செய்க மற்றும் அங்கீகாரத்தை உருவாக்குக


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. சரியான மாதிரியை கண்டுபிடித்தல்  
GPT-4o மற்றும் GPT-4o மினி போன்ற மாதிரிகள் இயற்கை மொழியைப் புரிந்து கொண்டு உருவாக்க முடியும், மேலும் Microsoft Foundry Models பகலில் Meta, Mistral, Cohere மற்றும் Microsoft ஆகியவற்றின் மாதிரிகளுடன் கிடைக்கின்றன.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. குறிப்பு வடிவமைப்பு  

"பெரிய மொழி மாடல்களின் மாயாஜாலம் என்னவென்றால், மிகப்பெரிய அளவிலான உரையில் இந்த கணிப்பு பிழையை குறைக்கும் பயிற்சியினால், மாடல்கள் இந்த கணிப்புகளுக்கு பயன்படும் கருத்துக்களை கற்றுக்கொள்கிறன. எடுத்துக்காட்டாக, அவை" (1)போன்ற கருத்துக்களை கற்றுக்கொள்கின்றன:

* எப்படி எழுத்துக்களை எழுதுவது
* எவ்வாறு இலக்கணப் பணிகள் செயல்படுகின்றன
* எப்படி மாற்றி சொல்வது
* எப்படி கேள்விகளுக்கு பதில் அளிக்குவது
* எப்படி உரையாடலை நடத்தியது
* பல மொழிகளில் எப்படி எழுதுவது
* எப்படி குறியீடு எழுதுவது
* இதர

#### பெரிய மொழி மாடலை எப்படி கட்டுப்படுத்துவது  
"ஒரு பெரிய மொழி மாதிரிக்கான அனைத்து உள்ளீடுகளில், மிக முக்கியமானது உரை குறிப்பு(1).

பெரிய மொழி மாடல்களை சில விதங்களில் உற்பத்தி செய்வதற்கு தூண்டிக் கொள்ளலாம்:

கையேடு: மாடலுக்கு நீங்கள் என்ன வேண்டுமோ சொல்லுங்கள்
முழுமை: நீங்கள் வேண்டியதற்கான தொடக்கத்தை மாடல் முடிக்க ஊக்குவியுங்கள்
காட்சிப்படுத்தல்: நீங்கள் வேண்டியதைக் காட்டுங்கள், அதில் ஒன்று:
குறிப்பு பகுதியில் சில எடுத்துக்காட்டுகள்
சிறந்த தொண்டு பயிற்சி தரவுத்தொகுப்பில் நூறுகள் அல்லது ஆயிரக்கணக்கான எடுத்துக்காட்டுகள்"



#### குறிப்புகளை உருவாக்க மூன்று அடிப்படை வழிகாட்டுதல்கள் உள்ளன:

**காட்டவும் சொல்லவும்**. நீங்கள் என்ன வேண்டுமென்று தெளிவாக கூறுங்கள், அதாவது கையேடுகள், எடுத்துக்காட்டுகள் அல்லது இரண்டின் கலவையால். நீங்கள் மாடலை ஒரு பொருட்களின் பட்டியலை வரிசைப்படுத்த வேண்டும் எனின் அல்லது ஒரு பத்தியை உணர்ச்சிப்படுத்த வேண்டும் எனில், அது என்ன வேண்டுமென தெளிவாகக் காட்டுங்கள்.

**உயர்தர தரவை வழங்குங்கள்**. நீங்கள் ஒரு வகைப்படுத்தியாளரை உருவாக்கவோ அல்லது மாடல் ஒரு முறைபாட்டை பின்பற்றவோ முயலின், போதுமான எடுத்துக்காட்டுகள் உள்ளதை உறுதிப்படுத்துங்கள். உங்கள் எடுத்துக்காட்டுகளை திருத்திப்பாருங்கள் — மாடல் பொதுவாக அடிப்படைக் எழுத்துப் பிழைகளை கண்டுபிடித்து பதில் அளிக்க நுட்பமுள்ளவையாவாலும், அதனை வரவழைக்கும் என்று கருதி பதிலை பாதிக்கக்கூடும்.

**உங்கள் அமைப்புகளை பரிசோதிக்கவும்.** வெப்பநிலை மற்றும் top_p அமைப்புகள் மாடல் பதில் உருவாக்கத்தில் எவ்வளவு உறுதியாக இருக்கிறது என்பதை கட்டுப்படுத்துகின்றன. ஒரே சரியான பதில் உள்ள கேள்விக்கு நீங்கள் கேட்கும் பொது, இதை குறைத்து அமைக்க வேண்டும். மாறுபட்ட பதில்களை எதிர்பார்க்கின், அதிகமாக அமைக்கலாம். இவற்றின் மீது மக்கள் பிரயோகிக்கும் முதல் தவறு இது 'ஞானம்' அல்லது 'புதுமை' கட்டுப்பாடுகள் என எண்ணுவது.


மூலம்: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. சமர்ப்பிக்கவும்!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### அதே அழைப்பை மீண்டும் செய்யும் போது, பெறுபவைகள் எப்படி ஒப்பிடப்படுகின்றன?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## உரையை சுருக்குக  
#### சவால்  
ஒரு உரை பாகத்தின் இறுதியில் 'tl;dr:' சேர்த்து உரையை சுருக்குக. அடிப்படையாக, மாதிரி எவ்வாறு பல பணிகளை கூடுதல் வழிமுறைகள் இல்லாமல் புரிந்து கொள்கிறது என்பதை கவனியுங்கள். tl;dr விட அதிக விளக்கமான கட்டளைகளுடன் முயற்சி செய்து மாதிரியின் நடத்தை மாற்றவும் நீங்கள் பெறும் சுருக்கத்தைக் தனிப்பயனாக்கவும் முடியும்(3).  

சமீபத்திய பணிகள் பெரும் உரை தொகுப்பில் முன்கல்வி செய்து, பின்னர் குறிப்பிட்ட பணிக்கு சிறுத்தலை மேற்கொண்டு பல NLP பணிகளிலும் மதிப்பீட்டிலும் பெரும் முன்னேற்றங்களை காட்சியளித்துள்ளன. பொதுவாக பணித் தொடர்பற்ற கட்டமைப்பில் இருந்தாலும், இந்த முறைக்கு ஆயிரக்கணக்கான அல்லது பத்தாயிரக்கணக்கான பணிசார் சிறுத்தல் தரவுத்தொகுப்புகள் தேவைப்படுகிறது. மாறாக, மனிதர்கள் பொதுவாக சில உதாரணங்களையோ அல்லது எளிய வழிமுறைகளைப் பயன்படுத்தி புதிய மொழிப்பணியை நிறைவேற்ற முடியும் - இது தற்போதைய NLP அமைப்புகள் இன்னும் பெரும்பான்மையாகச் சிரமப்படுகின்றன. இங்கே, மொழி மாதிரிகளை அளவிடல் அதிகரிப்பதால் பணித் தொடர்பற்ற, கொடுப்பளவு குறைந்த செயல்திறன் பெரிதும் மேம்படுவதாகவும், சில சமயம் முன்பணி சிறுத்தல் அணுகுமுறைகள் சார்ந்த முன்னணி செயல்திறனோடு போட்டியிடக்கூடியதாகவும் இருக்கிறது என்பதை காட்டுகிறோம்.  



சுருக்கமாக  


# பல பயன்படுத்தும் நிலைகளுக்கான பயிற்சிகள்  
1. உரையை சுருக்குக  
2. உரையை வகைப் பகுப்பாய்வு செய்யவும்  
3. புதிய தயாரிப்பு பெயர்களை உருவாக்கவும்


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## உரையை வகைப்படுத்துக  
#### சவால்  
வரையின் நேரத்தில் வழங்கப்பட்ட категорииகளுக்குள் பொருட்களை வகைப்படுத்துக. கீழ்காணும் உதாரணத்தில், நாம் வகைகள் மற்றும் வகைப்படுத்த வேண்டிய உரையை இரண்டையும் *playground_reference என்ற முன்மொழிவில் வழங்குகிறோம்.  

வாடிக்கையாளர் கேள்வி: வணக்கம், என் லேப்டாப் தட்டச்சுப்பலகையின் ஒரு தப்பான விசை சமீபத்தில் உடையானது, அதற்கான மாற்றம் தேவை:  

வகைப்படுத்தப்பட்ட பிரிவு:  


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## புதிய தயாரிப்புப் பெயர்களை உருவாக்கவும்
#### சவால்
உதாரண வார்த்தைகளிலிருந்து தயாரிப்புப் பெயர்களை உருவாக்கவும். இங்கு நாம்பாட்டில் நாம் பெயர்களை உருவாக்கப் போகும் தயாரிப்பின் பற்றிய தகவல்களையும் அடங்கியுள்ளது. மேலும், நாம் பெற விரும்பும் வடிவத்தை காட்ட ஒரு ஒத்த உதாரணத்தை வழங்குகிறோம். அதேபோல், சீரற்றத்தன்மையையும் இனோவேட்டிவ் பதில்களையும் அதிகப்படுத்த, வெப்பநிலை மதிப்பையும் உயர்த்தியுள்ளோம்.

தயாரிப்பு விவரம்: வீட்டு மில்க்ஷேக் இயந்திரம்
விதை வார்த்தைகள்: விரைவு, ஆரோக்கியமான, சுருக்கமான.
தயாரிப்புப் பெயர்கள்: HomeShaker, Fit Shaker, QuickShake, Shake Maker

தயாரிப்பு விவரம்: எந்த பாத அளவுக்கும் பொருந்தக்கூடிய செருப்பு ஜோடி.
விதை வார்த்தைகள்: சரிசெய்யக்கூடிய, பொருத்தமான, ஒம்னி-பிட்.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# குறிப்புகள்  
- [Openai குக்க்புக்](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry போர்டல்](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [உரை வகைப்படுத்த GPT மாதிரிகளை நுட்பமாகச் சீரமைப்பதற்கான சிறந்த பழக்கங்கள்](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# மேலதிக உதவிக்கு  
[OpenAI வணிகப்படுத்தல் குழு](AzureOpenAITeam@microsoft.com) 


# பங்களிப்பாளர்கள்
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
